# C1DB1B: Data Management for Decision-Making
## Workshop 2: Data Preparation, Hygiene & Pipeline Amalgamation
**Course**: C1DB1B (Master's Level / G1F, 7.5 ECTS)  
**Institution**: Högskolan i Borås  
**Module**: 2 – Data Preparation, Hygiene, Portfolio Transformations & Validation Governance  
**Prerequisite**: Module 1 Demographic Baseline (`VGR_Demographics_2015_2025.csv`)

---

### 🎯 Business Scenario: Building the Multi-Source Decision Pipeline
Welcome to Module 2! In Module 1, your team acted as healthcare decision support analysts for **Västra Götalandsregionen (VGR)**, curating the baseline demographic data and establishing candidate KPIs for allocating a **500 million SEK healthcare capital investment fund**.

However, demographic counts alone do not tell the full story. To make an evidence-based recommendation on where to place geriatric wards, primary care centers, and specialized clinics, VGR must combine demographic realities with **socioeconomic status, acute travel access, and public health lifestyle risks**.

In this workshop, you will progress from isolated demographic tables to a fully unified, audit-ready decision-support pipeline.

---

### 📚 Pedagogical Roadmap & Learning Approach
Every technical code cell in this notebook is preceded by a conceptual explanation connecting Python syntax to data governance and decision theory.

| Section | Focus Area | Technical & Decision Concepts |
| :--- | :--- | :--- |
| **Part 1** | 🐍 Foundations & Library Toolkit | Data objects (Scalars, Vectors, DataFrames), primary keys, and environment setup |
| **Part 2** | 🔗 Pipeline Amalgamation ("4+1" Join) | Merging demographic base with external health/socioeconomic data on `municipality_code` |
| **Part 3** | 🧹 Data Hygiene & Imputation Sensitivity | Catching `'..'` null markers; measuring statistical distortion of Mean vs. Median imputation |
| **Part 4** | 🎛️ Group Portfolio Choice Menu | Selecting one data-diagnosed transformation (Log, Winsorizing, Z-Score, Min-Max) |
| **Part 5** | 🔒 Validation Governance (20% Hold-Out) | Sequestering and freezing `VGR_Holdout_Test_Set.csv` to prevent data leakage for Module 3 |
| **Part 6** | 💼 Executive Audit Defense | Answering 3 structured business questions for Gateway 2 approval |


---
## 🐍 Part 1: Foundations & Python Toolkit

### Conceptual Connection: Data Objects in Decision Support Systems
In digital Decision Support Systems (DSS), we work with three primary data objects:
1. **Scalars (0D)**: Single values representing parameters, thresholds, or single metrics (e.g., `funding_pool = 500_000_000`).
2. **Vectors / Series (1D)**: Ordered 1-dimensional arrays representing a single attribute across entities (e.g., a Pandas `Series` of municipal travel times).
3. **Tables / DataFrames (2D)**: 2-dimensional grids where rows represent decision entities (VGR's 52 municipalities) and columns represent attributes or KPIs.

### Libraries Used in Module 2:
- **`pandas` (`pd`)**: Ingesting, restructuring, merging (`pd.merge`), and slicing DataFrames.
- **`numpy` (`np`)**: Mathematical vector operations, logarithmic transforms (`np.log1p`), and missing value constants (`np.nan`).
- **`scipy.stats`**: Statistical distribution diagnostics, skewness calculation, and winsorization.
- **`sklearn.model_selection`**: Deterministic hold-out splitting (`train_test_split`).
- **`matplotlib` & `seaborn`**: Visualizing distribution shapes and before/after transformation comparisons.


In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Library Imports
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

# Configure aesthetic themes for clean decision support graphics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("✅ Module 2 environment initialized successfully!")
print(f"   • Pandas version:       {pd.__version__}")
print(f"   • NumPy version:        {np.__version__}")


---
## 🔗 Part 2: Pipeline Amalgamation (The "4+1" Join)

### Conceptual Connection: Primary Keys & Decision Integrity
When combining data across multiple government agencies (e.g., Statistics Sweden SCB and Region Västra Götaland healthcare registries), datasets are created by different departments with differing naming conventions:
- Department A might record *"Göteborgs stad"*, while Department B records *"Göteborg"*.
- Merging on text names leads to catastrophic key-mismatches and dropped records!
- **Primary Key Standard**: We must always merge on standard official administrative keys—specifically the 4-digit **`municipality_code`** (e.g., `1480` for Göteborg, `1490` for Borås, `1427` for Sotenäs).

### Pandas Join Types & Decision Governance:
In `pd.merge(left_df, right_df, on='municipality_code', how=...)`:
- **`how='inner'`**: Keeps only entities present in **both** tables. Ideal when all indicators are mandatory.
- **`how='left'`**: Keeps all rows from the primary decision table (demographic base) and populates missing attributes from the secondary table.
- **Auditing Rule**: You must always audit row counts **before and after** merging. If your demographic table has 52 municipalities and your merged table has 47, you have lost 5 municipalities—which would exclude them from healthcare funding without administrative justification!


In [ ]:
# ==============================================================================
# Cell 2: Ingest Baseline Demographics & Extract Municipal Summary
# ==============================================================================

demo_file = "VGR_Demographics_2015_2025.csv"

if not os.path.exists(demo_file):
    raise FileNotFoundError(f"Missing '{demo_file}'. Ensure Module 1 outputs are in the workspace.")

print(f"📂 Ingesting Module 1 demographic baseline: '{demo_file}'...")
df_demo_raw = pd.read_csv(demo_file)

# 1. Filter out county total (region_code 14) and isolate combined gender ('totalt')
muni_mask = (df_demo_raw['region_code'] != 14) & (df_demo_raw['gender_label'] == 'totalt')

# 2. Extract Total Population (2025 & 2015) from 'TOT' age rows
df_tot = df_demo_raw[muni_mask & (df_demo_raw['age_code'] == 'TOT')][
    ['region_code', 'region_name', 'pop_2025', 'pop_2015']
].copy()
df_tot.rename(columns={
    'region_code': 'municipality_code',
    'region_name': 'municipality_name',
    'pop_2025': 'total_pop_2025',
    'pop_2015': 'total_pop_2015'
}, inplace=True)

# 3. Aggregate Elderly Cohort (Age >= 80) across single-year rows
df_ages = df_demo_raw[muni_mask & (df_demo_raw['age_code'] != 'TOT')].copy()
df_ages['age_int'] = pd.to_numeric(df_ages['age_code'], errors='coerce')

df_80p = df_ages[df_ages['age_int'] >= 80].groupby('region_code')[
    ['pop_2025', 'pop_2015']
].sum().reset_index()
df_80p.rename(columns={
    'region_code': 'municipality_code',
    'pop_2025': 'pop_80plus_2025',
    'pop_2015': 'pop_80plus_2015'
}, inplace=True)

# 4. Create the frozen Demographic Baseline DataFrame
df_baseline = pd.merge(df_tot, df_80p, on='municipality_code')
df_baseline['aging_ratio_2025'] = (df_baseline['pop_80plus_2025'] / df_baseline['total_pop_2025']) * 100
df_baseline['growth_80plus_pct'] = ((df_baseline['pop_80plus_2025'] - df_baseline['pop_80plus_2015']) / df_baseline['pop_80plus_2015']) * 100

print(f"✅ Demographic Baseline extracted: {len(df_baseline)} municipalities.")
df_baseline[['municipality_code', 'municipality_name', 'total_pop_2025', 'aging_ratio_2025', 'growth_80plus_pct']].head(4)


### 🎯 TASK 1: Perform Key-Based Pipeline Amalgamation
Below, you will load the external dataset (`VGR_External_Health_Socioeconomic.csv`), which contains:
- `median_income_ksek`: Median disposable household income (thousands of SEK).
- `avg_travel_time_min`: Average travel time in minutes to nearest acute emergency hospital.
- `lifestyle_risk_score`: Population public health lifestyle risk index (0–100).
- `care_utilization_rate`: Primary healthcare outpatient visits per capita.

Your goal is to execute a robust inner join on `municipality_code` and verify zero row loss.


In [ ]:
# ==============================================================================
# 🎯 TASK 1: Load External Dataset & Merge with Demographic Baseline
# ==============================================================================

ext_file = "VGR_External_Health_Socioeconomic.csv"

# Basic file existence check
if not os.path.exists(ext_file):
    raise FileNotFoundError(f"Missing external file: '{ext_file}'. Check file path.")

print(f"📂 Loading external dataset: '{ext_file}'...")
df_external = pd.read_csv(ext_file)

# Inspect external columns and shape
print(f"   External raw dimensions: {df_external.shape[0]} rows, {df_external.shape[1]} columns")

# ------------------------------------------------------------------------------
# Task 1.1: Merge df_baseline with df_external
# ------------------------------------------------------------------------------
# TODO: Use pd.merge() to join df_baseline and df_external on 'municipality_code'
# Hint: df_amalgamated = pd.merge(df_baseline, df_external, on='municipality_code', how='inner')
df_amalgamated = ...

# ------------------------------------------------------------------------------
# Task 1.2: Resolve overlapping columns if necessary (e.g. municipality_name)
# ------------------------------------------------------------------------------
# If municipality_name appears in both tables, Pandas appends _x and _y.
# Check if 'municipality_name_x' exists, clean it up:
# if 'municipality_name_x' in df_amalgamated.columns:
#     df_amalgamated['municipality_name'] = df_amalgamated['municipality_name_x']
#     df_amalgamated.drop(columns=['municipality_name_x', 'municipality_name_y'], inplace=True)

# ------------------------------------------------------------------------------
# Verification Output (Uncomment lines below once completed)
# ------------------------------------------------------------------------------
# print("=" * 65)
# print("PIPELINE AMALGAMATION AUDIT:")
# print(f"   • Baseline municipalities: {len(df_baseline)}")
# print(f"   • External municipalities: {len(df_external)}")
# print(f"   • Amalgamated rows:       {len(df_amalgamated)}")
# print(f"   • Rows preserved:          {len(df_amalgamated) / len(df_baseline) * 100:.1f}%")
# print("=" * 65)
# df_amalgamated.head(3)


---
## 🧹 Part 3: Data Hygiene & Imputation Sensitivity Analysis

### Conceptual Connection: Missing Value Noise & The Bias of Imputation
In raw administrative systems, null values are rarely stored as clean standard blanks. Government statistical registries frequently use:
- `'..'` (Data unavailable or withheld to preserve individual patient privacy in small towns).
- `'-'` (Zero or statistically non-significant sample).
- Empty spaces (`' '`) or `'N/A'`.

If these string markers are not caught, Pandas imports the entire numeric column as text (`object` dtype), completely blocking mathematical modeling!

### ⚠️ Mandatory Sensitivity Check 2: Imputation Sensitivity
When data values are missing, analysts must choose how to handle them:
1. **Listwise Deletion**: Dropping municipalities with missing data entirely.  
   *Risk*: Disproportionately deletes small rural municipalities, systematically biasing healthcare funding towards large wealthy cities!
2. **Mean Imputation**: Replacing missing values with the arithmetic average ($\bar{x}$).  
   *Risk*: **Artificially deflates standard deviation ($\sigma$)!** It clusters data at the center and falsely implies high certainty.
3. **Median Imputation**: Replacing missing values with the 50th percentile.  
   *Advantage*: Robust against extreme outliers, preserving distributional boundaries.

In the next section, you will programmatically compare the standard deviation ($\sigma$) when imputing with **Mean vs. Median**.


In [ ]:
# ==============================================================================
# Cell 4: Data Hygiene – Replacing Unrecognized Null Markers with np.nan
# ==============================================================================

# In administrative files, string null markers must be converted to IEEE np.nan
null_markers = ['..', '...', ' - ', '-', ' ', '', 'N/A', 'na']

# Replace markers across the amalgamated table
df_amalgamated = df_amalgamated.replace(null_markers, np.nan)

# Cast target columns to numeric floats
target_cols = ['avg_travel_time_min', 'lifestyle_risk_score', 'median_income_ksek', 'care_utilization_rate']
for col in target_cols:
    if col in df_amalgamated.columns:
        df_amalgamated[col] = pd.to_numeric(df_amalgamated[col], errors='coerce')

print("✅ Data Hygiene complete. Missing values count per attribute:")
print(df_amalgamated[target_cols].isna().sum())


### 🎯 TASK 2: Execute the Imputation Sensitivity Experiment
Select **`avg_travel_time_min`** as your test attribute. 

You must:
1. Compute the baseline standard deviation on raw observed data ($\sigma_{\text{raw}}$) using `.dropna()`.
2. Create a copy imputed with `.mean()` and compute $\sigma_{\text{mean}}$.
3. Create a copy imputed with `.median()` and compute $\sigma_{\text{median}}$.
4. Calculate the percentage change in dispersion:
   $$\Delta \sigma \% = \left(\frac{\sigma_{\text{imputed}} - \sigma_{\text{raw}}}{\sigma_{\text{raw}}}\right) \times 100\%$$


In [ ]:
# ==============================================================================
# 🎯 TASK 2: Imputation Sensitivity Analysis on Travel Time
# ==============================================================================

target_feature = 'avg_travel_time_min'

# 1. Baseline statistics on observed data (excluding nulls)
raw_series = df_amalgamated[target_feature].dropna()
sigma_raw  = raw_series.std()
mean_raw   = raw_series.mean()
median_raw = raw_series.median()

# ------------------------------------------------------------------------------
# Task 2.1: Impute missing values using Mean
# ------------------------------------------------------------------------------
# TODO: Use .fillna(mean_raw)
# series_mean_imputed = df_amalgamated[target_feature].fillna(mean_raw)
# sigma_mean = series_mean_imputed.std()
series_mean_imputed = ...
sigma_mean = ...

# ------------------------------------------------------------------------------
# Task 2.2: Impute missing values using Median
# ------------------------------------------------------------------------------
# TODO: Use .fillna(median_raw)
# series_median_imputed = df_amalgamated[target_feature].fillna(median_raw)
# sigma_median = series_median_imputed.std()
series_median_imputed = ...
sigma_median = ...

# ------------------------------------------------------------------------------
# Task 2.3: Compute Percentage Shrinkage in Dispersion (Delta Sigma %)
# ------------------------------------------------------------------------------
# delta_sigma_mean_pct   = ((sigma_mean - sigma_raw) / sigma_raw) * 100
# delta_sigma_median_pct = ((sigma_median - sigma_raw) / sigma_raw) * 100

# ------------------------------------------------------------------------------
# Verification Output (Uncomment lines below once completed)
# ------------------------------------------------------------------------------
# print("=" * 65)
# print(f"IMPUTATION SENSITIVITY AUDIT: {target_feature}")
# print("=" * 65)
# print(f"Observed Data (N={len(raw_series)}): Mean={mean_raw:.2f}, Median={median_raw:.2f}, Std Dev={sigma_raw:.2f}")
# print(f"Mean Imputation:   Std Dev={sigma_mean:.2f} (Change: {delta_sigma_mean_pct:.2f}%)")
# print(f"Median Imputation: Std Dev={sigma_median:.2f} (Change: {delta_sigma_median_pct:.2f}%)")
# print("=" * 65)


### 🔍 Strategic Decision Prompt 1: Imputation Impact
**Decision Context**: In VGR, municipalities with missing travel time data (`..`) are typically smaller rural communities (such as Karlsborg and Gullspång) where emergency services are outsourced or difficult to measure.

1. Did mean imputation increase or decrease the standard deviation ($\sigma$)? Why does filling missing values with the arithmetic mean compress statistical variance?
2. If healthcare planners use mean-imputed travel times to allocate emergency ambulance bases, how does this artificially disadvantage rural peripheral communities?

*(Write your analysis in the cell below)*


#### ✍️ [Student Answer for Prompt 1]:
*Write your response here...*


---
## 🎛️ Part 4: Preprocessing Choice Menu (Group Portfolio Constraint)

### Conceptual Connection: Non-Commensurate Scales & Group Diversity
In decision modeling, variables are measured on **non-commensurate scales** (e.g., income in 300,000 SEK vs. travel time in 45 minutes vs. aging ratios in 8%). If fed directly into analytical models, variables with huge numeric magnitudes (like income) dominate and drown out life-critical metrics (like travel access)!

### 👥 The Group Portfolio Diversity Rule
To ensure teams master multiple data science techniques and prevent uncritical code-copying, each member in your 4-person group must select **ONE distinct, data-diagnosed transformation**:

| Option | Advanced Technique | When to Select It | Mathematical Formula | Target Feature Example |
| :---: | :--- | :--- | :--- | :--- |
| **A** | **Log Transformation** | Substantially right-skewed data ($Skew > +1.0$) | $y = \ln(x + 1)$ | `median_income_ksek` |
| **B** | **Robust Winsorization** | Heavy outlier tails (extreme travel distances) | Cap at 5th & 95th percentiles | `avg_travel_time_min` |
| **C** | **Z-Score Normalization** | Approximately symmetric / bell-shaped data | $z = \frac{x - \mu}{\sigma}$ | `lifestyle_risk_score` |
| **D** | **Min-Max Feature Scaling** | Metrics requiring bounded $[0, 1]$ range | $x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ | `care_utilization_rate` |

---
### Diagnosing Distribution Skewness:
Before choosing, we calculate Fisher-Pearson skewness:
$$\text{Skewness} = \frac{N}{(N-1)(N-2)} \sum \left(\frac{x_i - \bar{x}}{\sigma}\right)^3$$
- $\text{Skew} > +1.0$: Strong right-skew $\rightarrow$ **Option A (Log)**
- Outliers beyond $3\sigma$ $\rightarrow$ **Option B (Winsorizing)**
- $-0.5 \le \text{Skew} \le +0.5$: Normal $\rightarrow$ **Option C (Z-Score)**
- Bounded continuous rates $\rightarrow$ **Option D (Min-Max)**


In [ ]:
# ==============================================================================
# Cell 6: Distribution Diagnostics – Skewness & Spread Assessment
# ==============================================================================

# Finalize clean imputation before transformation (using robust median)
for col in target_cols:
    df_amalgamated[col] = df_amalgamated[col].fillna(df_amalgamated[col].median())

print("=" * 65)
print("DISTRIBUTION DIAGNOSTIC REPORT:")
print("=" * 65)
for col in target_cols:
    sk = df_amalgamated[col].skew()
    q1 = df_amalgamated[col].quantile(0.25)
    q3 = df_amalgamated[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((df_amalgamated[col] < (q1 - 1.5 * iqr)) | (df_amalgamated[col] > (q3 + 1.5 * iqr))).sum()
    print(f"Feature: {col:<26} | Skewness: {sk:+.2f} | Outliers: {outliers}")
print("=" * 65)


### 🎯 TASK 3: Apply Your Assigned Group Portfolio Technique
In the cell below, **uncomment the single option** corresponding to your assigned role in your group portfolio.


In [ ]:
# ==============================================================================
# 🎯 TASK 3: The Group Portfolio Preprocessing Menu
# ==============================================================================
# UNCOMMENT EXACTLY ONE OPTION BASED ON YOUR GROUP ROLE:

# ------------------------------------------------------------------------------
# OPTION A: Log Transformation (For Right-Skewed Data: e.g. median_income_ksek)
# ------------------------------------------------------------------------------
# chosen_feature = 'median_income_ksek'
# transformed_col = 'income_log1p'
# df_amalgamated[transformed_col] = np.log1p(df_amalgamated[chosen_feature])
# technique_name = "Log Transformation [ln(x+1)]"

# ------------------------------------------------------------------------------
# OPTION B: Robust Winsorizing (For Outlier-Heavy Data: e.g. avg_travel_time_min)
# ------------------------------------------------------------------------------
# chosen_feature = 'avg_travel_time_min'
# transformed_col = 'travel_time_winsorized'
# # Cap bottom 5% and top 5% extreme values
# lower_cap = df_amalgamated[chosen_feature].quantile(0.05)
# upper_cap = df_amalgamated[chosen_feature].quantile(0.95)
# df_amalgamated[transformed_col] = df_amalgamated[chosen_feature].clip(lower=lower_cap, upper=upper_cap)
# technique_name = "Robust Winsorization (5th-95th percentile clipping)"

# ------------------------------------------------------------------------------
# OPTION C: Standard Z-Score Normalization (For Symmetric Data: e.g. lifestyle_risk_score)
# ------------------------------------------------------------------------------
# chosen_feature = 'lifestyle_risk_score'
# transformed_col = 'lifestyle_zscore'
# mu = df_amalgamated[chosen_feature].mean()
# sig = df_amalgamated[chosen_feature].std()
# df_amalgamated[transformed_col] = (df_amalgamated[chosen_feature] - mu) / sig
# technique_name = "Standard Z-Score Normalization [(x - μ) / σ]"

# ------------------------------------------------------------------------------
# OPTION D: Min-Max Feature Scaling (For Bounded [0,1] Range: e.g. care_utilization_rate)
# ------------------------------------------------------------------------------
# chosen_feature = 'care_utilization_rate'
# transformed_col = 'care_utilization_minmax'
# min_val = df_amalgamated[chosen_feature].min()
# max_val = df_amalgamated[chosen_feature].max()
# df_amalgamated[transformed_col] = (df_amalgamated[chosen_feature] - min_val) / (max_val - min_val)
# technique_name = "Min-Max Feature Scaling [(x - min) / (max - min)]"

# ------------------------------------------------------------------------------
# Verification Output (Uncomment lines below once completed)
# ------------------------------------------------------------------------------
# print("=" * 65)
# print(f"PORTFOLIO TECHNIQUE APPLIED: {technique_name}")
# print(f"   • Raw Feature:         {chosen_feature}")
# print(f"   • Transformed Feature:   {transformed_col}")
# print(f"   • Raw Skewness:        {df_amalgamated[chosen_feature].skew():+.2f}")
# print(f"   • Transformed Skewness:  {df_amalgamated[transformed_col].skew():+.2f}")
# print("=" * 65)
# df_amalgamated[['municipality_name', chosen_feature, transformed_col]].head(4)


### 🔍 Strategic Decision Prompt 2: Group Portfolio Justification
**Decision Context**: Executive healthcare boards demand transparent methodology. They need to understand why an analyst manipulated raw numbers before building allocation formulas.

1. Which preprocessing technique did you select, and how does your selection align with the statistical diagnostic of your variable?
2. How does enforcing a **Group Portfolio Constraint** (different techniques across team members) protect decision-making teams against groupthink and uncritical copying?

*(Write your analysis in the cell below)*


#### ✍️ [Student Answer for Prompt 2]:
*Write your response here...*


---
## 🔒 Part 5: Validation Governance (The 20% Hold-Out Lock)

### Conceptual Connection: Data Governance, Overfitting & Leakage
In Module 3 and 4, you will build statistical regression models and multi-criteria decision algorithms (MCDA). A pervasive failure mode in real-world policy modeling is **data leakage**:
- If an algorithm is tuned, calibrated, and evaluated on the exact same set of municipalities, it appears artificially accurate.
- When applied to new regions or future years, it collapses because it memorized noise rather than structural drivers.
- **The Golden Governance Rule**: Before any predictive modeling begins, we must sequester **exactly 20% of municipalities** into an isolated validation file (`VGR_Holdout_Test_Set.csv`) and lock it away. 
- The remaining **80%** (`VGR_Training_Model_Set.csv`) will be used to calibrate Module 3 models.


In [ ]:
# ==============================================================================
# 🎯 TASK 4: Sequester the 20% Hold-Out Validation Test Set
# ==============================================================================

# Ensure reproducible splitting using a fixed random seed (seed = 42)
test_ratio = 0.20
random_seed = 42

# ------------------------------------------------------------------------------
# Task 4.1: Split df_amalgamated into 80% Training and 20% Hold-Out Test sets
# ------------------------------------------------------------------------------
# TODO: Use train_test_split() from scikit-learn or df_amalgamated.sample()
# df_train, df_test = train_test_split(df_amalgamated, test_size=test_ratio, random_state=random_seed)
df_train = ...
df_test  = ...

# ------------------------------------------------------------------------------
# Task 4.2: Export both partitions to disk
# ------------------------------------------------------------------------------
# train_file = "VGR_Training_Model_Set.csv"
# test_file  = "VGR_Holdout_Test_Set.csv"
# df_train.to_csv(train_file, index=False, encoding='utf-8')
# df_test.to_csv(test_file, index=False, encoding='utf-8')

# ------------------------------------------------------------------------------
# Verification Output (Uncomment lines below once completed)
# ------------------------------------------------------------------------------
# print("=" * 65)
# print("VALIDATION GOVERNANCE AUDIT (THE 20% HOLD-OUT LOCK):")
# print("=" * 65)
# print(f"Total Amalgamated Entities:       {len(df_amalgamated)}")
# print(f"Training / Modeling Set (80%):    {len(df_train)} municipalities -> Saved to '{train_file}'")
# print(f"Sequestered Hold-Out Set (20%):   {len(df_test)} municipalities  -> Saved to '{test_file}'")
# print(f"Zero Data Leakage Check:          Intersection = {len(set(df_train['municipality_code']).intersection(set(df_test['municipality_code'])))} entities")
# print("=" * 65)
# print("\nMunicipalities Sequestered for Hold-Out Testing:")
# print(df_test[['municipality_code', 'municipality_name', 'total_pop_2025']].to_string(index=False))


### 🔍 Strategic Decision Prompt 3: Data Governance & Ethics
**Decision Context**: Public policy decisions must withstand legal and ethical scrutiny.

1. Why must the 20% hold-out test set be sequestered *before* entering Module 3 (Statistical Modeling), rather than splitting after models are constructed?
2. If an executive board discovered that a predictive healthcare model was evaluated on data it had already seen during calibration, what are the organizational and ethical consequences?

*(Write your analysis in the cell below)*


#### ✍️ [Student Answer for Prompt 3]:
*Write your response here...*


---
## 🎯 Part 6: Gateway 2 Checklist & Submission Artifacts

### What You Mastered in Module 2:
1. **Pipeline Amalgamation**: Joining disparate departmental sources via official numeric primary keys (`municipality_code`) without dropping records.
2. **Data Hygiene & Sensitivity**: Catching real-world missing data markers (`..`) and quantifying how mean vs. median imputation alters dispersion.
3. **Group Portfolio Preprocessing**: Selecting and defending an advanced transformation (Log, Winsorizing, Z-Score, Min-Max) matching your dataset's statistical profile.
4. **Validation Governance**: Sequestering and freezing a clean 20% hold-out test set to ensure rigorous out-of-sample testing in Modules 3 and 4.

---
### 🏁 Submission Instructions:
1. Download your completed notebook: **File $\rightarrow$ Download** (as `.ipynb`).
2. Verify that `VGR_Holdout_Test_Set.csv` and `VGR_Training_Model_Set.csv` are generated in your working directory.
3. Submit your notebook and 2-page Data Hygiene Audit Report to Canvas for Gateway 2 evaluation!
